# TradeSense · Component 03 · Stage D · Notebook D1b
### Diagnostic: how much can ANY simple predictor beat persistence under the frozen MAE metric?

**Why this notebook exists.** D1 showed that a flattened-window MLP (B3) does not beat persistence (B1), and that 78% of the target values do not change in the next 100 ms interval. Under mean absolute error the best guess for a value that usually does not change is "no change", so persistence is hard to beat. Before spending about 10 GPU hours on the TGNN (D2), this notebook measures how much room there is.

**What it measures (validation month, July, and a training sample for fitting only):**
1. Per rank (bid and ask 1-25): how often a node changes, how often a change follows a change, how often a change reverses the previous one.
2. A *binned conditional-median predictor* (fitted on training windows): the best lookup-table guess given node, size of the last change and number of recent changes.
3. A *shared-node L1 network* (one small network applied to every node, trained on 10% of training windows for 3 epochs): does local history carry any information at all?
4. How well the next-interval *occurrence* of a change can be predicted (AUC), which matters for anomaly scoring even if MAE does not move.

**What it does NOT do.** It changes no frozen rule: the metric stays MAE, the splits stay as they are, nothing about M0/M1 is altered. It opens no August, September or October day. The decision rule for reading the result is written in cell 9 before the numbers exist. Result files go into a new `d1b` folder inside your existing `TGNN_StageD_v1` (nothing there is overwritten).

**Use a T4 GPU runtime.** Expected time: about 20 to 30 minutes. If Colab stops, press *Run all* again.

## 1 · Settings (nothing to edit unless your D1 folder has another name)

In [ ]:
import os, gc, sys, json, time, shutil, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from IPython.display import display
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets')); assert BASE.is_dir(), f'Drive folder not found: {BASE}'
STAGEC_NAME = 'TGNN_StageC_GraphFeatures_v2'
D1_NAME = 'TGNN_StageD_v1'                     # the finished D1 folder; D1b writes only into its new subfolder d1b
FIT_WINDOWS = 300_000                          # random training windows used to fit the lookup table
SHARED_FRAC = 0.10; SHARED_EPOCHS = 3; SHARED_BATCH = 512; SEED = 0
print('Stage C:', STAGEC_NAME, '| D1 folder:', D1_NAME)

Mounted at /content/drive
Stage C: TGNN_StageC_GraphFeatures_v2 | D1 folder: TGNN_StageD_v1


## 2 · Libraries (written to the Colab working folder, not to Drive)

`staged_lib.py` is the exact D1 library (same hash is checked in cell 4). `staged_diag_lib.py` holds the diagnostic code.

In [ ]:
%%writefile staged_lib.py
"""TradeSense C03 - Stage D library (D1: loader, leak guards, baselines B1-B4, B3 MLP, resumable trainer).
Reads ONLY the frozen Stage C tensors. Never writes to Stage C. Test days are refused until a freeze file exists."""
import os, json, time, math, hashlib, shutil
from pathlib import Path
import numpy as np
import torch, torch.nn as nn

SEQ = 50; NN = 50; NCH = 6; GCH = 3; NT = 51; LOGQ = 2; GLS = 0; GIMB = 1
SYMS = ['BTCUSDT', 'ETHUSDT']
SPLITS = {'train': ['2024-05-01', '2024-06-01'], 'val': ['2024-07-01'], 'test': ['2024-08-01', '2024-09-01', '2024-10-01']}
def keys_of(split): return [f'{s}_{d}' for d in SPLITS[split] for s in SYMS]

# ---------------------------------------------------------------- small helpers
def sha256_file(p, chunk=16 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()
def _j(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, Path): return str(o)
    raise TypeError(type(o))
def canonical_sha(obj): return hashlib.sha256(json.dumps(obj, sort_keys=True, default=_j).encode()).hexdigest()
def dump_json(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True); tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, indent=1, default=_j)); os.replace(tmp, path)
def put_json(obj, path):
    """write once; if a different file already exists it is NEVER replaced: the new content goes to <name>.rN.json"""
    path = Path(path)
    if path.exists():
        if json.loads(path.read_text()) == json.loads(json.dumps(obj, default=_j)): return path
        n = 1
        while path.with_name(f'{path.stem}.r{n}{path.suffix}').exists(): n += 1
        path = path.with_name(f'{path.stem}.r{n}{path.suffix}')
    dump_json(obj, path); return path
def check_frozen(path, cfg):
    """first call writes the frozen config; later calls must match exactly, otherwise resuming is refused"""
    path = Path(path)
    if not path.exists(): dump_json({'config': cfg, 'sha256': canonical_sha(cfg)}, path); return 'written'
    old = json.loads(path.read_text())
    if old['sha256'] != canonical_sha(cfg): raise RuntimeError(f'frozen config differs from the saved one ({path}); do not change settings or code while resuming a run')
    return 'matches'

# ---------------------------------------------------------------- the loader (all tensors in one block, optionally on the GPU)
class Bank:
    def __init__(self, stagec_dir, keys, device='cpu', freeze_file=None, chunk=100_000):
        self.dir = Path(stagec_dir) / 'sets'; self.keys = list(keys); self.device = device; self.summ = {}; total = 0
        for k in self.keys:
            s = json.loads((self.dir / k / 'set_summary.json').read_text()); self.summ[k] = s
            if s['split'] == 'test' and not (freeze_file and Path(freeze_file).exists()):
                raise PermissionError(f'{k} is a TEST day: refused because the freeze file does not exist yet')
            total += int(s['n_intervals'])
        self.nodes = torch.empty((total, NN, NCH), dtype=torch.float16, device=device); self.glob = torch.empty((total, GCH), dtype=torch.float16, device=device)
        self.off = {}; wins = []; sid = []; o = 0
        for i, k in enumerate(self.keys):
            arr = np.load(self.dir / k / 'nodes.npy', mmap_mode='r'); g = np.load(self.dir / k / 'glob.npy', mmap_mode='r'); n = arr.shape[0]
            assert arr.shape[1:] == (NN, NCH) and g.shape == (n, GCH), f'{k}: unexpected tensor shape {arr.shape} {g.shape}'
            for a in range(0, n, chunk):
                b = min(a + chunk, n); self.nodes[o + a:o + b] = torch.from_numpy(np.array(arr[a:b])).to(device)
            self.glob[o:o + n] = torch.from_numpy(np.array(g)).to(device)
            w = np.load(self.dir / k / 'win_end.npy'); wins.append(w.astype(np.int64) + o); sid.append(np.full(len(w), i, np.int16)); self.off[k] = (o, n); o += n; del arr, g
        self.win = np.concatenate(wins); self.win_set = np.concatenate(sid); self.ar = torch.arange(-SEQ + 1, 1, device=device)
    def gb(self): return (self.nodes.numel() * 2 + self.glob.numel() * 2) / 1e9
    def batch(self, g):
        """g: LongTensor of GLOBAL end indices t. Input = intervals t-49..t (inclusive). Target = interval t+1 only."""
        idx = g[:, None] + self.ar
        xn = self.nodes[idx].float(); xg = self.glob[idx].float()
        yn = self.nodes[g + 1][:, :, LOGQ].float(); yg = self.glob[g + 1][:, GLS].float()
        y = torch.cat([yn, yg[:, None]], 1); last = torch.cat([xn[:, -1, :, LOGQ], xg[:, -1, GLS:GLS + 1]], 1)
        return xn, xg, y, last

# ---------------------------------------------------------------- evaluation harness (per-window losses are kept for the later bootstrap)
@torch.no_grad()
def evaluate(bank, predict, bs=4096, extra=None):
    n = len(bank.win); loss = np.zeros(n, np.float32); chg = np.zeros(n, np.float32); nch = np.zeros(n, np.int16); spr = np.zeros(n, np.float32); ex = []
    for a in range(0, n, bs):
        b = min(a + bs, n); g = torch.from_numpy(bank.win[a:b]).to(bank.device); xn, xg, y, last = bank.batch(g)
        err = (predict(xn, xg, last) - y).abs(); m = (y != last)
        loss[a:b] = err.mean(1).cpu().numpy(); chg[a:b] = (err * m).sum(1).cpu().numpy(); nch[a:b] = m.sum(1).cpu().numpy(); spr[a:b] = err[:, NN].cpu().numpy()
        if extra is not None: ex.append(extra(xn, xg, last).cpu().numpy())
    out = {'loss': loss, 'chg_sum': chg, 'n_chg': nch, 'spr': spr}
    if extra is not None: out['extra'] = np.concatenate(ex)
    return out
def summarize(bank, res):
    rows = {}
    for i, k in enumerate(bank.keys):
        m = bank.win_set == i
        rows[k] = {'n_windows': int(m.sum()), 'mae_all': float(res['loss'][m].mean()), 'mae_changed': float(res['chg_sum'][m].sum() / max(int(res['n_chg'][m].sum()), 1)),
                   'share_changed': float(res['n_chg'][m].sum() / (NT * max(int(m.sum()), 1))), 'mae_spread': float(res['spr'][m].mean())}
    return rows
def average_results(rs):
    """seeds are averaged FIRST (mean of the per-seed losses of each window), exactly as in the frozen rule"""
    return {'loss': np.mean([r['loss'] for r in rs], 0).astype(np.float32), 'chg_sum': np.mean([r['chg_sum'] for r in rs], 0).astype(np.float32),
            'n_chg': rs[0]['n_chg'], 'spr': np.mean([r['spr'] for r in rs], 0).astype(np.float32)}
def save_result(res, bank, path):
    path = Path(path)
    if path.exists(): raise FileExistsError(path)
    tmp = path.with_name(path.stem + '.tmp.npz'); np.savez_compressed(tmp, win=bank.win, win_set=bank.win_set, **{k: v for k, v in res.items() if k != 'extra'}); os.replace(tmp, path)

# ---------------------------------------------------------------- baselines
def pred_b1(xn, xg, last): return last
def ema_weights(span, device='cpu'):
    a = 2.0 / (span + 1.0); j = torch.arange(SEQ - 1, -1, -1, dtype=torch.float32, device=device)       # time index 0 = oldest, 49 = latest
    w = (1 - a) ** j; return w / w.sum()
SPANS = [1, 2, 3, 5, 8, 13, 21, 34, 50]
def make_b2(span_node, span_spread):
    def f(xn, xg, last):
        wn = ema_weights(span_node, xn.device); ws = ema_weights(span_spread, xn.device)
        pn = torch.einsum('t,btn->bn', wn, xn[:, :, :, LOGQ]); ps = torch.einsum('t,bt->b', ws, xg[:, :, GLS])
        return torch.cat([pn, ps[:, None]], 1)
    return f
@torch.no_grad()
def tune_b2(bank, bs=4096):
    """one span for node targets, one span for the spread target, chosen on validation only"""
    sn = np.zeros(len(SPANS)); ss = np.zeros(len(SPANS)); n = len(bank.win)
    for a in range(0, n, bs):
        b = min(a + bs, n); g = torch.from_numpy(bank.win[a:b]).to(bank.device); xn, xg, y, last = bank.batch(g)
        for i, sp in enumerate(SPANS):
            p = make_b2(sp, sp)(xn, xg, last); e = (p - y).abs(); sn[i] += float(e[:, :NN].sum()); ss[i] += float(e[:, NN].sum())
    sn /= n * NN; ss /= n; return {'span_node': SPANS[int(sn.argmin())], 'span_spread': SPANS[int(ss.argmin())], 'val_mae_node_by_span': dict(zip(SPANS, sn.tolist())), 'val_mae_spread_by_span': dict(zip(SPANS, ss.tolist()))}
def pred_b4(xn, xg, last):
    """B4: the z-score centre as a naive predictor. Spread target -> window mean (z = 0). Node targets are not part of B4 (persistence is used only so that a tensor exists; B4 is judged on the spread column only)."""
    return torch.cat([last[:, :NN], xg[:, :, GLS].mean(1, keepdim=True)], 1)
def b4_score(xn, xg, last):
    """rolling z-score anomaly score of spread and top-5 imbalance (max of the two absolute z), saved for later fusion/anomaly use"""
    def z(v): return ((v[:, -1] - v.mean(1)) / (v.std(1) + 1e-3)).abs()
    return torch.maximum(z(xg[:, :, GLS]), z(xg[:, :, GIMB]))

# ---------------------------------------------------------------- B3 MLP (the same output rule is used for every learned model: prediction = value at t + learned change)
class MLP(nn.Module):
    def __init__(s, h):
        super().__init__(); s.net = nn.Sequential(nn.Linear(SEQ * NN * NCH + SEQ * GCH, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, NT))
        nn.init.zeros_(s.net[-1].weight); nn.init.zeros_(s.net[-1].bias)           # starts exactly at persistence
    def forward(s, xn, xg, last): return last + s.net(torch.cat([xn.flatten(1), xg.flatten(1)], 1))
def make_mlp(cfg): return MLP(cfg['hidden'])
GRID = [{'hidden': h, 'lr': lr} for h in (64, 128) for lr in (1e-3, 3e-4)]      # the 4 pre-listed configurations

# ---------------------------------------------------------------- resumable trainer (also used by D2 for the TGNN)
def _load_latest(out, model, opt):
    for p in sorted(out.glob('resume_e*.pt'), reverse=True):
        try: ck = torch.load(p, map_location='cpu', weights_only=False)
        except Exception: continue
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); torch.set_rng_state(ck['rng_cpu'])
        if ck.get('rng_cuda') is not None and torch.cuda.is_available(): torch.cuda.set_rng_state_all(ck['rng_cuda'])
        return ck['state'], p.name
    return None, None
def fit(make_model, cfg, bank_tr, bank_va, out_dir, seed, frac=0.10, max_epochs=20, patience=3, bs=256, device='cpu', log=print, _halt_after=None):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    if (out / 'done.json').exists(): return json.loads((out / 'done.json').read_text())
    assert all(bank_tr.summ[k]['split'] == 'train' for k in bank_tr.keys), 'training bank contains a non-training set'
    torch.manual_seed(seed); model = make_model(cfg).to(device); opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    state = {'epoch': 0, 'best': float('inf'), 'best_epoch': 0, 'bad': 0, 'hist': []}
    st, name = _load_latest(out, model, opt)
    if st is not None: state = st; log(f'  resumed from {name} (epoch {state["epoch"]})')
    n = len(bank_tr.win)
    while state['epoch'] < max_epochs and state['bad'] < patience:
        e = state['epoch'] + 1; t0 = time.time(); rng = np.random.default_rng([seed, e]); pick = rng.permutation(n)[:int(n * frac)]
        model.train(); tot = torch.zeros((), device=device); cnt = 0
        for a in range(0, len(pick), bs):
            g = torch.from_numpy(bank_tr.win[pick[a:a + bs]]).to(device); xn, xg, y, last = bank_tr.batch(g)
            loss = (model(xn, xg, last) - y).abs().mean(); opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += loss.detach() * len(g); cnt += len(g)
        t_train = time.time() - t0; model.eval(); res = evaluate(bank_va, lambda xn, xg, last: model(xn, xg, last)); val = float(res['loss'].mean())
        torch.save(model.state_dict(), out / f'weights_e{e:03d}.pt')
        state['epoch'] = e; state['hist'].append({'epoch': e, 'train_mae': float(tot / max(cnt, 1)), 'val_mae': val, 'train_seconds': round(t_train, 1), 'epoch_seconds': round(time.time() - t0, 1)})
        if val < state['best'] - 1e-9: state['best'] = val; state['best_epoch'] = e; state['bad'] = 0
        else: state['bad'] += 1
        tmp = out / f'resume_e{e:03d}.pt.tmp'
        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'rng_cpu': torch.get_rng_state(), 'rng_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None, 'state': state}, tmp); os.replace(tmp, out / f'resume_e{e:03d}.pt')
        for p in out.glob('resume_e*.pt'):                                  # keep the two newest optimiser states; weights of every epoch stay
            if int(p.stem[8:]) < e - 1: p.unlink()
        with open(out / 'log.jsonl', 'a') as f: f.write(json.dumps(state['hist'][-1]) + '\n')
        log(f'  epoch {e}: train {state["hist"][-1]["train_mae"]:.5f} | val {val:.5f} | best {state["best"]:.5f} (e{state["best_epoch"]}) | {state["hist"][-1]["epoch_seconds"]} s')
        if _halt_after is not None and e >= _halt_after: return None
    done = {'cfg': cfg, 'seed': seed, 'best_val': state['best'], 'best_epoch': state['best_epoch'], 'epochs_run': state['epoch'], 'history': state['hist'], 'n_params': int(sum(p.numel() for p in model.parameters())),
            'mean_epoch_seconds': float(np.mean([h['epoch_seconds'] for h in state['hist']])), 'frac': frac, 'max_epochs': max_epochs, 'patience': patience, 'batch_size': bs}
    dump_json(done, out / 'done.json'); return done
def load_best(make_model, cfg, out_dir, device='cpu'):
    d = json.loads((Path(out_dir) / 'done.json').read_text()); m = make_model(cfg).to(device); m.load_state_dict(torch.load(Path(out_dir) / f'weights_e{d["best_epoch"]:03d}.pt', map_location=device)); m.eval(); return m, d

# ---------------------------------------------------------------- synthetic Stage C folder (for the code tests only)
def make_mock_stagec(root, n_int=2600, seed=0, bad_gap=(1200, 1290)):
    root = Path(root); sets = root / 'sets'; (root / 'config').mkdir(parents=True, exist_ok=True); rng = np.random.default_rng(seed)
    dump_json({'scaler': {'mock': True}, 'sha256': canonical_sha({'mock': True})}, root / 'config' / 'scaler_train_only.json')
    for split in ('train', 'val', 'test'):
        for day in SPLITS[split]:
            for sym in SYMS:
                k = f'{sym}_{day}'; d = sets / k; d.mkdir(parents=True, exist_ok=True)
                lq = np.zeros((n_int, NN), np.float32); cur = rng.normal(size=NN)
                for i in range(n_int):
                    ch = rng.random(NN) < 0.25; cur = np.where(ch, 0.9 * cur + rng.normal(scale=0.5, size=NN), cur); lq[i] = cur
                nodes = rng.normal(size=(n_int, NN, NCH)).astype(np.float32); nodes[:, :, 0] = np.where(np.arange(NN) < 25, -1, 1); nodes[:, :, LOGQ] = lq
                gl = rng.normal(size=(n_int, GCH)).astype(np.float32); gl[1:, GLS] = np.where(rng.random(n_int - 1) < 0.2, rng.normal(size=n_int - 1), 0); gl[:, GLS] = np.maximum.accumulate(np.zeros(n_int)) + np.cumsum(np.where(rng.random(n_int) < .1, rng.normal(scale=.3, size=n_int), 0))
                flags = np.zeros(n_int, np.uint8); flags[:SEQ + 20] = 16; flags[bad_gap[0]:bad_gap[1]] = 8
                c = np.concatenate([[0], np.cumsum((flags != 0).astype(np.int64))]); t = np.arange(SEQ, n_int - 1); win = t[(c[t + 2] - c[t - SEQ]) == 0].astype(np.int32)
                np.save(d / 'nodes.npy', nodes.astype(np.float16)); np.save(d / 'glob.npy', gl.astype(np.float16)); np.save(d / 'flags.npy', flags); np.save(d / 'win_end.npy', win)
                dump_json({'sym': sym, 'day': day, 'split': split, 'n_intervals': n_int, 'n_windows': int(len(win)), 'scaler_sha256': canonical_sha({'mock': True}), 'hashes': {f: sha256_file(d / f) for f in ('nodes.npy', 'glob.npy', 'flags.npy', 'win_end.npy')}}, d / 'set_summary.json')
    return root

# ---------------------------------------------------------------- E-tests (code on synthetic data)
def unit_tests(tmp):
    tmp = Path(tmp); out = []
    def add(i, nm, ok, det=''): out.append((i, nm, bool(ok), str(det)))
    root = make_mock_stagec(tmp / 'sc', n_int=2600); fz = tmp / 'freeze.json'
    # E01 test days are refused before the freeze file exists
    try: Bank(root, keys_of('test')[:1], freeze_file=fz); refused = False
    except PermissionError: refused = True
    dump_json({'x': 1}, fz); ok_after = True
    try: Bank(root, keys_of('test')[:1], freeze_file=fz)
    except Exception: ok_after = False
    fz.unlink(); add('E01', 'test days refused until the freeze file exists', refused and ok_after, f'refused={refused} allowed_after={ok_after}')
    btr = Bank(root, keys_of('train'), device='cpu'); bva = Bank(root, keys_of('val'), device='cpu')
    # E02 exact window indexing against direct array access
    rng = np.random.default_rng(1); worst = 0.0; okidx = True
    for i in rng.choice(len(btr.win), 40, replace=False):
        gt = int(btr.win[i]); k = btr.keys[int(btr.win_set[i])]; o, n = btr.off[k]; t = gt - o
        raw = np.load(btr.dir / k / 'nodes.npy'); rg = np.load(btr.dir / k / 'glob.npy')
        xn, xg, y, last = btr.batch(torch.tensor([gt]))
        okidx &= np.array_equal(xn[0].numpy(), raw[t - 49:t + 1].astype(np.float32)) and np.array_equal(xg[0].numpy(), rg[t - 49:t + 1].astype(np.float32))
        okidx &= np.array_equal(y[0, :NN].numpy(), raw[t + 1, :, LOGQ].astype(np.float32)) and float(y[0, NN]) == float(rg[t + 1, GLS]) and np.array_equal(last[0, :NN].numpy(), raw[t, :, LOGQ].astype(np.float32))
    add('E02', 'input = intervals t-49..t, target = interval t+1 only (40 random windows, exact)', okidx)
    # E03 the future cannot leak: overwrite interval t+1 and t+2 and check the input does not change
    gt = int(btr.win[100]); a = btr.batch(torch.tensor([gt]))[0].clone(); y0 = btr.batch(torch.tensor([gt]))[2].clone()
    sv = btr.nodes[gt + 1:gt + 3].clone(); btr.nodes[gt + 1:gt + 3] = 77.0; b = btr.batch(torch.tensor([gt]))
    same_x = torch.equal(a, b[0]); y_changed = not torch.equal(y0, b[2]); btr.nodes[gt + 1:gt + 3] = sv; add('E03', 'changing t+1 and t+2 does not change the input, only the target', same_x and y_changed)
    # E04 no window touches a flagged interval, and no window crosses a day boundary
    okf = True; okb = True
    for i, k in enumerate(btr.keys):
        fl = np.load(btr.dir / k / 'flags.npy'); w = np.load(btr.dir / k / 'win_end.npy'); okf &= all((fl[t - SEQ:t + 2] == 0).all() for t in w[::7]); okb &= int(w.min()) >= SEQ and int(w.max()) + 1 < btr.off[k][1]
    add('E04', 'windows contain no flagged interval and stay inside their day', okf and okb)
    # E05 baselines
    bs_ = 512; res1 = evaluate(bva, pred_b1, bs=bs_); man = []
    for i in rng.choice(len(bva.win), 30, replace=False):
        xn, xg, y, last = bva.batch(torch.tensor([int(bva.win[i])])); man.append((i, float((last - y).abs().mean())))
    okb1 = all(abs(res1['loss'][i] - v) < 1e-6 for i, v in man)
    res2 = evaluate(bva, make_b2(1, 1)); w = ema_weights(8); okb2 = np.allclose(res2['loss'], res1['loss'], atol=1e-6) and abs(float(w.sum()) - 1) < 1e-6 and bool((w >= 0).all()) and bool(w[-1] == w.max())
    add('E05', 'B1 equals a hand computation; B2 with span 1 equals B1; EMA weights are valid and causal', okb1 and okb2)
    # E06 changed-node mask by hand
    xn = torch.zeros(1, SEQ, NN, NCH); xg = torch.zeros(1, SEQ, GCH); last = torch.zeros(1, NT); y = torch.zeros(1, NT); y[0, 3] = 1.0; y[0, 50] = 2.0; p = torch.zeros(1, NT); err = (p - y).abs(); m = (y != last)
    add('E06', 'changed-target mask and per-window sums', int(m.sum()) == 2 and float((err * m).sum()) == 3.0)
    # E07 MLP starts exactly at persistence and can learn
    torch.manual_seed(0); mlp = MLP(16); xn, xg, y, last = btr.batch(torch.tensor(btr.win[:8])); p0 = mlp(xn, xg, last); okz = torch.equal(p0, last)
    opt = torch.optim.Adam(mlp.parameters(), lr=1e-3); l0 = float((mlp(xn, xg, last) - y).abs().mean().detach())
    for _ in range(60): l = (mlp(xn, xg, last) - y).abs().mean(); opt.zero_grad(); l.backward(); opt.step()
    add('E07', 'B3 starts at persistence and its loss falls on a small fit', okz and float(l.detach()) < l0 * 0.9, f'{l0:.4f}->{float(l.detach()):.4f}')
    # E08 resume gives exactly the same result as an uninterrupted run
    cfg = {'hidden': 8, 'lr': 1e-3}; quiet = lambda *a, **k: None
    d1 = fit(make_mlp, cfg, btr, bva, tmp / 'r_full', seed=5, frac=0.2, max_epochs=3, patience=9, bs=64, log=quiet)
    fit(make_mlp, cfg, btr, bva, tmp / 'r_part', seed=5, frac=0.2, max_epochs=3, patience=9, bs=64, log=quiet, _halt_after=1)
    d2 = fit(make_mlp, cfg, btr, bva, tmp / 'r_part', seed=5, frac=0.2, max_epochs=3, patience=9, bs=64, log=quiet)
    sa = torch.load(tmp / 'r_full' / 'weights_e003.pt'); sb = torch.load(tmp / 'r_part' / 'weights_e003.pt')
    add('E08', 'a stopped-and-resumed run equals an uninterrupted run (weights and validation history)', all(torch.equal(sa[k], sb[k]) for k in sa) and [h['val_mae'] for h in d1['history']] == [h['val_mae'] for h in d2['history']])
    # E09 seed averaging and per-set summary by hand
    r = [{'loss': np.array([1., 2.], np.float32), 'chg_sum': np.array([2., 4.], np.float32), 'n_chg': np.array([1, 2], np.int16), 'spr': np.array([0., 1.], np.float32)}, {'loss': np.array([3., 4.], np.float32), 'chg_sum': np.array([4., 6.], np.float32), 'n_chg': np.array([1, 2], np.int16), 'spr': np.array([2., 3.], np.float32)}]
    av = average_results(r); add('E09', 'seeds are averaged per window before any statistic', np.allclose(av['loss'], [2, 3]) and np.allclose(av['chg_sum'], [3, 5]) and np.allclose(av['spr'], [1, 2]))
    # E10 sampler: deterministic per (seed, epoch), training bank holds only training sets, validation bank only validation
    p1 = np.random.default_rng([3, 2]).permutation(1000)[:100]; p2 = np.random.default_rng([3, 2]).permutation(1000)[:100]
    add('E10', 'epoch sample is deterministic per (seed, epoch); training bank = training sets only; validation bank = validation sets only', np.array_equal(p1, p2) and all(btr.summ[k]['split'] == 'train' for k in btr.keys) and all(bva.summ[k]['split'] == 'val' for k in bva.keys))
    # E11 frozen config
    try: check_frozen(tmp / 'fz.json', {'a': 1}); check_frozen(tmp / 'fz.json', {'a': 1}); refused = False; check_frozen(tmp / 'fz.json', {'a': 2})
    except RuntimeError: refused = True
    add('E11', 'a changed frozen config is refused', refused)
    # E12 put_json never overwrites
    put_json({'a': 1}, tmp / 'p.json'); put_json({'a': 2}, tmp / 'p.json'); add('E12', 'put_json keeps the first file and writes a changed one as .r1', json.loads((tmp / 'p.json').read_text()) == {'a': 1} and (tmp / 'p.r1.json').exists())
    # E13 B4 and tune_b2 run and give sane numbers
    tb = tune_b2(bva); sc_ = evaluate(bva, pred_b4, extra=b4_score); add('E13', 'B2 tuning and B4 run; B4 score is finite and non-negative', tb['span_node'] in SPANS and np.isfinite(sc_['extra']).all() and (sc_['extra'] >= 0).all())
    return out

Writing staged_lib.py


In [ ]:
%%writefile staged_diag_lib.py
"""D1b diagnostic library: how much can ANY simple predictor beat persistence under the frozen MAE metric?
Validation/training windows only. Nothing here changes a frozen rule."""
import json, time, math
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import staged_lib as S
from staged_lib import SEQ, NN, NT, LOGQ, GLS, GIMB

NH = 8                                   # number of recent changes shown to the shared-node net
NCLS = 5                                 # change classes: neg-large, neg-small, zero, pos-small, pos-large
NCNT = 3                                 # number of changes in the last 5 intervals: 0, 1, 2+

# ---------------------------------------------------------------- per-window quantities (all in scaled units)
def window_deltas(xn, xg, y, last):
    """returns: dn_hist (B,NH,50) the last NH changes of log_qty per node, d_last (B,50), d_next (B,50), sp_hist (B,NH), sp_last (B,), sp_next (B,)"""
    q = xn[:, :, :, LOGQ]; dq = q[:, 1:] - q[:, :-1]                    # (B,49,50)
    s = xg[:, :, GLS]; ds = s[:, 1:] - s[:, :-1]                        # (B,49)
    d_next = y[:, :NN] - last[:, :NN]; sp_next = y[:, NN] - last[:, NN]
    return dq[:, -NH:], dq[:, -1], d_next, ds[:, -NH:], ds[:, -1], sp_next, dq[:, -5:], ds[:, -5:]
def change_class(d, edge):
    """0 neg-large, 1 neg-small, 2 zero, 3 pos-small, 4 pos-large; zero is EXACTLY zero (values are stored, not computed, so unchanged quantities are bit-identical)"""
    a = d.abs(); c = torch.full(d.shape, 2, dtype=torch.long, device=d.device)
    c = torch.where(d > 0, torch.where(a >= edge, torch.full_like(c, 4), torch.full_like(c, 3)), c)
    c = torch.where(d < 0, torch.where(a >= edge, torch.zeros_like(c), torch.ones_like(c)), c)
    return c
def count_class(d5):
    k = (d5 != 0).sum(1); return torch.clamp(k, max=NCNT - 1)

# ---------------------------------------------------------------- 1. per-node change statistics
@torch.no_grad()
def node_stats(bank, windows, bs=2048):
    z = lambda: torch.zeros(NN, dtype=torch.float64)
    A = {k: z() for k in ('n', 'chg', 'prev_chg', 'chg_given_prev', 'chg_given_noprev', 'both', 'reverse', 'abs_b1')}; sp = {'n': 0, 'chg': 0, 'abs_b1': 0.0}
    for a in range(0, len(windows), bs):
        g = torch.from_numpy(windows[a:a + bs]).to(bank.device); xn, xg, y, last = bank.batch(g)
        _, dl, dn, _, sl, sn, _, _ = window_deltas(xn, xg, y, last)
        ch = (dn != 0); pv = (dl != 0)
        A['n'] += torch.full((NN,), len(g), dtype=torch.float64); A['chg'] += ch.sum(0).double().cpu(); A['prev_chg'] += pv.sum(0).double().cpu()
        A['chg_given_prev'] += (ch & pv).sum(0).double().cpu(); A['chg_given_noprev'] += (ch & ~pv).sum(0).double().cpu()
        both = ch & pv; A['both'] += both.sum(0).double().cpu(); A['reverse'] += (both & (torch.sign(dn) == -torch.sign(dl))).sum(0).double().cpu(); A['abs_b1'] += dn.abs().sum(0).double().cpu()
        sp['n'] += len(g); sp['chg'] += int((sn != 0).sum()); sp['abs_b1'] += float(sn.abs().sum())
    n = A['n']; out = {'p_change': (A['chg'] / n).numpy(), 'p_change_given_prev': (A['chg_given_prev'] / A['prev_chg'].clamp(min=1)).numpy(),
                       'p_change_given_noprev': (A['chg_given_noprev'] / (n - A['prev_chg']).clamp(min=1)).numpy(), 'p_reverse_given_both': (A['reverse'] / A['both'].clamp(min=1)).numpy(),
                       'mae_b1': (A['abs_b1'] / n).numpy(), 'spread_p_change': sp['chg'] / sp['n'], 'spread_mae_b1': sp['abs_b1'] / sp['n']}
    return out
def rank_groups(v):
    """average the 25 bid and 25 ask ranks into rank buckets 1-3, 4-10, 11-25 (bid and ask averaged)"""
    v = np.asarray(v); r = {}
    for nm, (a, b) in {'rank 1-3': (0, 3), 'rank 4-10': (3, 10), 'rank 11-25': (10, 25)}.items(): r[nm] = float((v[a:b].mean() + v[25 + a:25 + b].mean()) / 2)
    return r

# ---------------------------------------------------------------- 2. binned conditional-median predictor (fit on TRAIN windows, scored on validation)
@torch.no_grad()
def fit_edge(bank, windows, bs=2048):
    vals = []
    for a in range(0, len(windows), bs):
        g = torch.from_numpy(windows[a:a + bs]).to(bank.device); xn, xg, y, last = bank.batch(g); dl = window_deltas(xn, xg, y, last)[1]; vals.append(dl.abs()[dl != 0].cpu())
    v = torch.cat(vals); return float(v.median()) if len(v) else 1.0
@torch.no_grad()
def _bins(xn, xg, y, last, edge):
    dh, dl, dn, sh, sl, sn, d5, s5 = window_deltas(xn, xg, y, last)
    cn = change_class(dl, edge) * NCNT + count_class(d5)[...]                     # (B,50)  class index inside a node
    cn = cn + torch.arange(NN, device=xn.device)[None, :] * (NCLS * NCNT)        # node-specific bin
    cs = change_class(sl[:, None], edge)[:, 0] * NCNT + count_class(s5[:, :, None])[:, 0]
    return cn, cs, dn, sn
@torch.no_grad()
def fit_binned(bank, windows, edge, bs=2048):
    nb = NN * NCLS * NCNT; node_vals = [[] for _ in range(nb)]; sp_vals = [[] for _ in range(NCLS * NCNT)]; allb, alld, sb, sd = [], [], [], []
    for a in range(0, len(windows), bs):
        g = torch.from_numpy(windows[a:a + bs]).to(bank.device); xn, xg, y, last = bank.batch(g); cn, cs, dn, sn = _bins(xn, xg, y, last, edge)
        allb.append(cn.flatten().cpu().numpy()); alld.append(dn.flatten().cpu().numpy()); sb.append(cs.cpu().numpy()); sd.append(sn.cpu().numpy())
    b = np.concatenate(allb); d = np.concatenate(alld); node_med = np.zeros(nb, np.float32); node_n = np.bincount(b, minlength=nb)
    order = np.argsort(b, kind='stable'); b_s = b[order]; d_s = d[order]; starts = np.searchsorted(b_s, np.arange(nb + 1))
    for i in range(nb):
        if starts[i + 1] > starts[i]: node_med[i] = np.median(d_s[starts[i]:starts[i + 1]])
    sbb = np.concatenate(sb); sdd = np.concatenate(sd); sp_med = np.zeros(NCLS * NCNT, np.float32)
    for i in range(NCLS * NCNT):
        m = sbb == i
        if m.any(): sp_med[i] = np.median(sdd[m])
    return {'node_med': node_med, 'spread_med': sp_med, 'edge': edge, 'node_n': node_n}
@torch.no_grad()
def eval_binned(bank, windows, model, bs=2048):
    nm = torch.from_numpy(model['node_med']).to(bank.device); sm = torch.from_numpy(model['spread_med']).to(bank.device); R = _acc()
    for a in range(0, len(windows), bs):
        g = torch.from_numpy(windows[a:a + bs]).to(bank.device); xn, xg, y, last = bank.batch(g); cn, cs, dn, sn = _bins(xn, xg, y, last, model['edge'])
        _add(R, nm[cn] - dn, sm[cs] - sn, dn, sn)
    return _fin(R)

# ---------------------------------------------------------------- accumulate errors (all-target MAE with equal weight per target value; changed-target MAE)
def _acc(): return {'abs_all': 0.0, 'n_all': 0, 'abs_chg': 0.0, 'n_chg': 0, 'abs_b1_all': 0.0, 'abs_b1_chg': 0.0, 'sp_abs': 0.0, 'sp_b1': 0.0, 'n_w': 0, 'rank': {}}
def _add(R, en, es, dn, sn):
    en = en.abs(); es = es.abs(); m = (dn != 0); ms = (sn != 0)
    R['abs_all'] += float(en.sum() + es.sum()); R['n_all'] += en.numel() + es.numel(); R['abs_b1_all'] += float(dn.abs().sum() + sn.abs().sum())
    R['abs_chg'] += float((en * m).sum() + (es * ms).sum()); R['n_chg'] += int(m.sum() + ms.sum()); R['abs_b1_chg'] += float((dn.abs() * m).sum() + (sn.abs() * ms).sum())
    R['sp_abs'] += float(es.sum()); R['sp_b1'] += float(sn.abs().sum()); R['n_w'] += len(sn)
    for nm, (a, b) in {'rank 1-3': (0, 3), 'rank 4-10': (3, 10), 'rank 11-25': (10, 25)}.items():
        idx = list(range(a, b)) + list(range(25 + a, 25 + b)); e = en[:, idx]; d = dn[:, idx].abs(); r = R['rank'].setdefault(nm, [0.0, 0.0, 0])
        r[0] += float(e.sum()); r[1] += float(d.sum()); r[2] += e.numel()
def _fin(R):
    out = {'mae_all': R['abs_all'] / R['n_all'], 'mae_all_b1': R['abs_b1_all'] / R['n_all'], 'mae_changed': R['abs_chg'] / max(R['n_chg'], 1), 'mae_changed_b1': R['abs_b1_chg'] / max(R['n_chg'], 1),
           'mae_spread': R['sp_abs'] / R['n_w'], 'mae_spread_b1': R['sp_b1'] / R['n_w'], 'n_windows': R['n_w']}
    out['rel_all_pct'] = 100 * (1 - out['mae_all'] / out['mae_all_b1']); out['rel_changed_pct'] = 100 * (1 - out['mae_changed'] / out['mae_changed_b1']); out['rel_spread_pct'] = 100 * (1 - out['mae_spread'] / max(out['mae_spread_b1'], 1e-12))
    out['by_rank'] = {k: {'mae': v[0] / v[2], 'mae_b1': v[1] / v[2], 'rel_pct': 100 * (1 - v[0] / max(v[1], 1e-12))} for k, v in R['rank'].items()}
    return out

# ---------------------------------------------------------------- 3. shared-node L1 network (one small net applied to every node; asks "is there any local information at all?")
class SharedNet(nn.Module):
    def __init__(s, h=128):
        super().__init__(); s.emb = nn.Embedding(NN, 8)
        s.node = nn.Sequential(nn.Linear(NH + 6 + 8 + 3 + NH, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 2))
        s.spread = nn.Sequential(nn.Linear(NH + 3, 64), nn.ReLU(), nn.Linear(64, 1))
        nn.init.zeros_(s.node[-1].weight); nn.init.zeros_(s.node[-1].bias); nn.init.zeros_(s.spread[-1].weight); nn.init.zeros_(s.spread[-1].bias)
    def forward(s, xn, xg, y, last):
        dh, dl, dn, sh, sl, sn, _, _ = window_deltas(xn, xg, y, last); B = xn.shape[0]; cl = lambda v: torch.clamp(v, -10, 10)
        node_now = xn[:, -1]                                                      # (B,50,6) channels at t
        x = torch.cat([cl(dh).permute(0, 2, 1), node_now, s.emb.weight[None].expand(B, -1, -1), xg[:, -1][:, None, :].expand(-1, NN, -1), cl(sh)[:, None, :].expand(-1, NN, -1)], 2)
        o = s.node(x); dpred = o[..., 0]; logit = o[..., 1]
        sp = s.spread(torch.cat([cl(sh), xg[:, -1]], 1))[:, 0]
        return dpred, logit, sp, dn, sn
def train_shared(btr, bva, frac=0.10, epochs=3, bs=512, device='cpu', seed=0, log=print, lr=1e-3):
    torch.manual_seed(seed); net = SharedNet().to(device); opt = torch.optim.Adam(net.parameters(), lr=lr); n = len(btr.win); hist = []
    bce = nn.BCEWithLogitsLoss()
    for e in range(1, epochs + 1):
        t0 = time.time(); rng = np.random.default_rng([seed, e]); pick = rng.permutation(n)[:int(n * frac)]; net.train(); tot = 0.0; cnt = 0
        for a in range(0, len(pick), bs):
            g = torch.from_numpy(btr.win[pick[a:a + bs]]).to(device); xn, xg, y, last = btr.batch(g); dp, lg, sp, dn, sn = net(xn, xg, y, last)
            loss = (dp - dn).abs().mean() + (sp - sn).abs().mean() / NN + 0.2 * bce(lg, (dn != 0).float()); opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); tot += float(loss.detach()) * len(g); cnt += len(g)
        hist.append({'epoch': e, 'train_loss': tot / max(cnt, 1), 'seconds': round(time.time() - t0, 1)}); log(f'  shared net epoch {e}: loss {hist[-1]["train_loss"]:.5f} | {hist[-1]["seconds"]} s')
    return net, hist
@torch.no_grad()
def eval_shared(net, bank, windows, bs=1024, auc_max=1_500_000, seed=0):
    net.eval(); R = _acc(); rng = np.random.default_rng(seed); scores = []; labels = []; kept = 0
    for a in range(0, len(windows), bs):
        g = torch.from_numpy(windows[a:a + bs]).to(bank.device); xn, xg, y, last = bank.batch(g); dp, lg, sp, dn, sn = net(xn, xg, y, last)
        _add(R, dp - dn, sp - sn, dn, sn)
        if kept < auc_max:
            s_ = lg.flatten().cpu().numpy(); l_ = (dn != 0).flatten().cpu().numpy(); scores.append(s_); labels.append(l_); kept += len(s_)
    out = _fin(R); out['auc_change'] = auc(np.concatenate(labels), np.concatenate(scores)); out['base_rate_change'] = float(np.concatenate(labels).mean()); return out
def auc(labels, scores):
    """rank-based AUC (Mann-Whitney) with average ranks for ties"""
    labels = np.asarray(labels).astype(bool); scores = np.asarray(scores, dtype=np.float64); n1 = labels.sum(); n0 = len(labels) - n1
    if n1 == 0 or n0 == 0: return float('nan')
    order = np.argsort(scores, kind='mergesort'); s = scores[order]; ranks = np.empty(len(s)); i = 0
    while i < len(s):
        j = i
        while j + 1 < len(s) and s[j + 1] == s[i]: j += 1
        ranks[i:j + 1] = (i + j) / 2 + 1; i = j + 1
    r = np.empty(len(s)); r[order] = ranks; return float((r[labels].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))
@torch.no_grad()
def eval_b1_reference(bank, windows, bs=2048):
    R = _acc()
    for a in range(0, len(windows), bs):
        g = torch.from_numpy(windows[a:a + bs]).to(bank.device); xn, xg, y, last = bank.batch(g); _, _, dn, _, _, sn, _, _ = window_deltas(xn, xg, y, last); _add(R, dn, sn, dn, sn)
    return _fin(R)

# ---------------------------------------------------------------- tests on synthetic data (can the diagnostic actually SEE a signal when one exists?)
def make_signal_bank(tmp, kind, n_int=3000, seed=0):
    """a Stage-C-shaped folder where log_qty follows a known rule: 'none' = random sparse jumps (nothing to learn), 'reverse' = every change is undone on the next interval (strong, learnable)"""
    root = S.make_mock_stagec(tmp, n_int=n_int, seed=seed); rng = np.random.default_rng(seed + 1)
    for split in ('train', 'val'):
        for day in S.SPLITS[split]:
            for sym in S.SYMS:
                d = Path(root) / 'sets' / f'{sym}_{day}'; nodes = np.load(d / 'nodes.npy').astype(np.float32); q = np.zeros((n_int, NN), np.float32); cur = rng.normal(size=NN).astype(np.float32); prev = cur.copy()
                for i in range(n_int):
                    if kind == 'reverse':
                        if i % 2 == 0: q[i] = cur
                        else: q[i] = cur + 1.0       # a bump at odd steps
                        # odd steps are +1 above the base, even steps return to the base: the change at odd t is +1 and is undone at t+1
                    else:
                        ch = rng.random(NN) < 0.2; cur = np.where(ch, cur + rng.normal(scale=0.5, size=NN), cur); q[i] = cur
                nodes[:, :, LOGQ] = q; np.save(d / 'nodes.npy', nodes.astype(np.float16))
                sm = json.loads((d / 'set_summary.json').read_text()); sm['hashes']['nodes.npy'] = S.sha256_file(d / 'nodes.npy'); S.dump_json(sm, d / 'set_summary.json')
    return root
def unit_tests(tmp):
    tmp = Path(tmp); out = []
    def add(i, nm, ok, det=''): out.append((i, nm, bool(ok), str(det)))
    r_none = make_signal_bank(tmp / 'none', 'none'); r_rev = make_signal_bank(tmp / 'rev', 'reverse')
    btr = S.Bank(r_none, S.keys_of('train')); bva = S.Bank(r_none, S.keys_of('val'))
    # F01 deltas and classes by hand
    xn = torch.zeros(1, SEQ, NN, S.NCH); xn[0, -1, 3, LOGQ] = 2.0; xn[0, -2, 3, LOGQ] = 0.5; xg = torch.zeros(1, SEQ, S.GCH); y = torch.zeros(1, NT); last = torch.zeros(1, NT)
    dh, dl, dn, sh, sl, sn, d5, s5 = window_deltas(xn, xg, y, last); cls = change_class(dl, 1.0)
    add('F01', 'last change, class and count of changes by hand', abs(float(dl[0, 3]) - 1.5) < 1e-6 and int(cls[0, 3]) == 4 and int(cls[0, 4]) == 2 and int(count_class(d5)[0, 3]) == 2, f'{float(dl[0,3])} class {int(cls[0,3])}')
    # F02 on pure noise nothing can beat persistence by much (binned and shared)
    wtr = btr.win[:6000]; wva = bva.win[:3000]; edge = fit_edge(btr, wtr); mb = fit_binned(btr, wtr, edge); eb = eval_binned(bva, wva, mb)
    net, _ = train_shared(btr, bva, frac=0.5, epochs=2, bs=128, log=lambda *a, **k: None); es = eval_shared(net, bva, wva)
    add('F02', 'no-signal data: binned and shared-net improvement over persistence stays below 3%', eb['rel_all_pct'] < 3 and es['rel_all_pct'] < 3, f"binned {eb['rel_all_pct']:.2f}% shared {es['rel_all_pct']:.2f}%")
    # F03 strong-signal data: both diagnostics must see it
    ttr = S.Bank(r_rev, S.keys_of('train')); tva = S.Bank(r_rev, S.keys_of('val')); wtr2 = ttr.win[:6000]; wva2 = tva.win[:3000]; e2 = fit_edge(ttr, wtr2); m2 = fit_binned(ttr, wtr2, e2); b2 = eval_binned(tva, wva2, m2)
    n2, _ = train_shared(ttr, tva, frac=0.5, epochs=3, bs=128, log=lambda *a, **k: None, lr=3e-3); s2 = eval_shared(n2, tva, wva2)
    add('F03', 'strong-signal data (every change is undone next step): binned improves a lot, shared net improves clearly', b2['rel_all_pct'] > 30 and s2['rel_all_pct'] > 5, f"binned {b2['rel_all_pct']:.1f}% shared {s2['rel_all_pct']:.1f}%")
    # F04 AUC function
    a1 = auc([0, 0, 1, 1], [0.1, 0.2, 0.8, 0.9]); a2 = auc([0, 1, 0, 1], [0.5, 0.5, 0.5, 0.5]); rng = np.random.default_rng(0); a3 = auc(rng.integers(0, 2, 20000), rng.normal(size=20000))
    add('F04', 'AUC: perfect = 1, all tied = 0.5, random about 0.5', a1 == 1.0 and abs(a2 - 0.5) < 1e-9 and abs(a3 - 0.5) < 0.02)
    # F05 statistics: persistence reference equals node_stats mae_b1
    ns = node_stats(btr, wtr); ref = eval_b1_reference(btr, wtr); add('F05', 'persistence error from two code paths agrees', abs(ref['mae_all_b1'] - (ns['mae_b1'].sum() + ns['spread_mae_b1']) / NT) < 1e-6)
    return out

Writing staged_diag_lib.py


## 3 · Safety checks: device, Stage C, finished D1, library hash equal to D1, frozen configuration

In [ ]:
import torch
sys.path.insert(0, str(Path.cwd())); import staged_lib as S, staged_diag_lib as D
LIB_SHA = S.sha256_file(Path('staged_lib.py')); DLIB_SHA = S.sha256_file(Path('staged_diag_lib.py'))
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; GPU = torch.cuda.get_device_name(0) if DEV == 'cuda' else None
SC = BASE / STAGEC_NAME; assert (SC / 'sets').is_dir(), f'Stage C folder not found: {SC}'
D1 = BASE / D1_NAME; rep1 = json.loads((D1 / 'reports' / 'd1_report.json').read_text()); assert rep1['verdict'] == 'D1 COMPLETE', 'D1 is not complete'
assert rep1['library_sha256'][:12] == LIB_SHA[:12], f'staged_lib.py differs from the one used in D1 ({rep1["library_sha256"][:12]} vs {LIB_SHA[:12]})'
OUT = D1 / 'd1b'; OUT.mkdir(exist_ok=True); (OUT / 'config').mkdir(exist_ok=True)
FROZEN = {'stagec': STAGEC_NAME, 'd1': D1_NAME, 'fit_windows': FIT_WINDOWS, 'shared_frac': SHARED_FRAC, 'shared_epochs': SHARED_EPOCHS, 'shared_batch': SHARED_BATCH, 'seed': SEED, 'library_sha256': LIB_SHA, 'diag_library_sha256': DLIB_SHA, 'metric': 'MAE, equal weight per target value (unchanged)'}
print('frozen config:', S.check_frozen(OUT / 'config' / 'frozen_config_d1b.json', FROZEN), '| device', DEV, GPU, '| D1b folder:', OUT.relative_to(BASE))
FREEZE_FILE = D1 / 'config' / 'freeze_d3.json'; assert not FREEZE_FILE.exists()

frozen config: written | device cuda Tesla T4 | D1b folder: TGNN_StageD_v1/d1b


## 4 · Tests: D1 library tests plus diagnostic tests (can the diagnostic see a signal when one exists, and stay quiet when none exists?)

In [ ]:
import tempfile
t0 = time.time()
with tempfile.TemporaryDirectory() as t_: ER1 = S.unit_tests(t_)
with tempfile.TemporaryDirectory() as t_: ER2 = D.unit_tests(t_)
ER = ER1 + ER2; display(pd.DataFrame(ER, columns=['id', 'test', 'pass', 'detail']))
assert all(r[2] for r in ER), 'A test failed: stop here and send me this table.'
print(f'tests: {sum(r[2] for r in ER)}/{len(ER)} passed in {time.time()-t0:.0f} s')

,id,test,pass,detail
0,E01,test days refused until the freeze file exists,True,refused=True allowed_after=True
1,E02,"input = intervals t-49..t, target = interval t...",True,
2,E03,changing t+1 and t+2 does not change the input...,True,
3,E04,windows contain no flagged interval and stay i...,True,
4,E05,B1 equals a hand computation; B2 with span 1 e...,True,
5,E06,changed-target mask and per-window sums,True,
6,E07,B3 starts at persistence and its loss falls on...,True,0.0820->0.0329
7,E08,a stopped-and-resumed run equals an uninterrup...,True,
8,E09,seeds are averaged per window before any stati...,True,
9,E10,"epoch sample is deterministic per (seed, epoch...",True,


tests: 18/18 passed in 27 s


## 5 · Load training and validation tensors (test days stay closed)

In [ ]:
t0 = time.time(); BTR = S.Bank(SC, S.keys_of('train'), device=DEV, freeze_file=FREEZE_FILE); BVA = S.Bank(SC, S.keys_of('val'), device=DEV, freeze_file=FREEZE_FILE)
try: S.Bank(SC, S.keys_of('test')[:1], device='cpu', freeze_file=FREEZE_FILE); TEST_REFUSED = False
except PermissionError: TEST_REFUSED = True
assert TEST_REFUSED; print(f'train {len(BTR.win):,} windows | validation {len(BVA.win):,} windows | test days refused: {TEST_REFUSED} | {time.time()-t0:.0f} s')
rng = np.random.default_rng(SEED); W_FIT = BTR.win[rng.choice(len(BTR.win), min(FIT_WINDOWS, len(BTR.win)), replace=False)]
VAL = {'pooled': rng.permutation(BVA.win)}
for i, k in enumerate(BVA.keys): VAL[k] = rng.permutation(BVA.win[BVA.win_set == i])

train 3,453,034 windows | validation 1,725,989 windows | test days refused: True | 49 s


## 6 · How often do nodes change, and what follows a change?

In [ ]:
t0 = time.time(); ST = {k: D.node_stats(BVA, w) for k, w in VAL.items() if k == 'pooled'}['pooled']
rows = []
for nm in ('rank 1-3', 'rank 4-10', 'rank 11-25'):
    rows.append({'ranks (bid and ask averaged)': nm, 'P(change at t+1)': D.rank_groups(ST['p_change'])[nm], 'P(change | changed at t)': D.rank_groups(ST['p_change_given_prev'])[nm], 'P(change | no change at t)': D.rank_groups(ST['p_change_given_noprev'])[nm],
                 'P(reverses | both changed)': D.rank_groups(ST['p_reverse_given_both'])[nm], 'persistence MAE': D.rank_groups(ST['mae_b1'])[nm]})
rows.append({'ranks (bid and ask averaged)': 'spread', 'P(change at t+1)': ST['spread_p_change'], 'persistence MAE': ST['spread_mae_b1']})
STAT_DF = pd.DataFrame(rows); display(STAT_DF.round(4)); print(f'{time.time()-t0:.0f} s')
print('Reading guide: if P(change) is below 0.5 for a node, "no change" is already the median guess (MAE-optimal) unless some information makes a change likely.')

,ranks (bid and ask averaged),P(change at t+1),P(change | changed at t),P(change | no change at t),P(reverses | both changed),persistence MAE
0,rank 1-3,0.1801,0.4149,0.1285,0.5951,0.0801
1,rank 4-10,0.1764,0.4127,0.1260,0.7066,0.1270
2,rank 11-25,0.2507,0.4462,0.1856,0.7287,0.1835
3,spread,0.0026,NaN,NaN,NaN,0.0267


3 s
Reading guide: if P(change) is below 0.5 for a node, "no change" is already the median guess (MAE-optimal) unless some information makes a change likely.


## 7 · Binned conditional-median predictor (fit on training windows, scored on validation)

In [ ]:
t0 = time.time(); EDGE = D.fit_edge(BTR, W_FIT); BM = D.fit_binned(BTR, W_FIT, EDGE); BIN = {k: D.eval_binned(BVA, w, BM) for k, w in VAL.items()}
print(f'edge {EDGE:.3f} | bins with data: {(BM["node_n"] > 0).sum()} of {len(BM["node_n"])} | {time.time()-t0:.0f} s')
display(pd.DataFrame({k: {'persistence all': v['mae_all_b1'], 'binned all': v['mae_all'], 'gain all %': v['rel_all_pct'], 'persistence changed': v['mae_changed_b1'], 'binned changed': v['mae_changed'], 'gain changed %': v['rel_changed_pct'], 'spread gain %': v['rel_spread_pct']} for k, v in BIN.items()}).T.round(5))

edge 0.278 | bins with data: 550 of 750 | 10 s


,persistence all,binned all,gain all %,persistence changed,binned changed,gain changed %,spread gain %
pooled,0.15275,0.15278,-0.02127,0.70346,0.70207,0.19648,16.63120
BTCUSDT_2024-07-01,0.04560,0.04548,0.28054,0.21728,0.21589,0.63749,18.88544
ETHUSDT_2024-07-01,0.25996,0.26015,-0.07425,1.15852,1.15714,0.11906,9.65498


## 8 · Shared-node L1 network and change-occurrence AUC

In [ ]:
t0 = time.time(); NET, HIST = D.train_shared(BTR, BVA, frac=SHARED_FRAC, epochs=SHARED_EPOCHS, bs=SHARED_BATCH, device=DEV, seed=SEED)
SHR = {k: D.eval_shared(NET, BVA, w) for k, w in VAL.items()}; print(f'{time.time()-t0:.0f} s')
display(pd.DataFrame({k: {'persistence all': v['mae_all_b1'], 'shared net all': v['mae_all'], 'gain all %': v['rel_all_pct'], 'persistence changed': v['mae_changed_b1'], 'shared net changed': v['mae_changed'], 'gain changed %': v['rel_changed_pct'], 'spread gain %': v['rel_spread_pct'], 'AUC (will a node change?)': v['auc_change'], 'base rate of change': v['base_rate_change']} for k, v in SHR.items()}).T.round(5))
RK = pd.DataFrame({k: {'persistence MAE': v['mae_b1'], 'shared net MAE': v['mae'], 'gain %': v['rel_pct']} for k, v in SHR['pooled']['by_rank'].items()}).T; display(RK.round(5))

  shared net epoch 1: loss 0.26230 | 3.1 s
  shared net epoch 2: loss 0.25229 | 2.6 s
  shared net epoch 3: loss 0.25097 | 2.6 s
29 s


,persistence all,shared net all,gain all %,persistence changed,shared net changed,gain changed %,spread gain %,AUC (will a node change?),base rate of change
pooled,0.15275,0.15147,0.83581,0.70346,0.68677,2.37238,27.49943,0.73189,0.21906
BTCUSDT_2024-07-01,0.04560,0.04446,2.51322,0.21728,0.20619,5.10239,29.36180,0.72313,0.21476
ETHUSDT_2024-07-01,0.25996,0.25855,0.54135,1.15852,1.13659,1.89314,21.73593,0.74416,0.22843


,persistence MAE,shared net MAE,gain %
rank 1-3,0.08013,0.07958,0.69615
rank 4-10,0.12699,0.12536,1.28129
rank 11-25,0.18349,0.18244,0.57464


## 9 · Reading rule (written before the numbers) and verdict

In [ ]:
# ---- decision rule, written before the numbers were seen (pooled validation) ----
g_all = max(BIN['pooled']['rel_all_pct'], SHR['pooled']['rel_all_pct']); g_chg = max(BIN['pooled']['rel_changed_pct'], SHR['pooled']['rel_changed_pct']); AUC = SHR['pooled']['auc_change']
if g_all < 1.0 and g_chg < 1.0: HEAD = 'NO HEADROOM'; MSG = 'Persistence is essentially unbeatable under MAE at 100 ms with local information. A TGNN can only pass H1/H2 if graph structure carries information that no local model sees. Expect H1/H2 to fail; that result would be reported as failed.'
elif g_all < 5.0: HEAD = 'SMALL HEADROOM'; MSG = f'The best simple model gains {g_all:.2f}% (all targets) and {g_chg:.2f}% (changed targets). A 5% pass needs the TGNN to beat these simple models clearly.'
else: HEAD = 'HEADROOM EXISTS'; MSG = f'A simple model already gains {g_all:.2f}% (all targets). The 5% rule is reachable; D2 is worth running.'
OCC = 'predictable' if AUC >= 0.70 else ('weakly predictable' if AUC >= 0.60 else 'not predictable')
print(f'verdict: {HEAD}\n{MSG}\nWhether a node will change in the next interval is {OCC} (AUC {AUC:.3f}); this matters for anomaly scoring even when MAE does not move.')

verdict: SMALL HEADROOM
The best simple model gains 0.84% (all targets) and 2.37% (changed targets). A 5% pass needs the TGNN to beat these simple models clearly.
Whether a node will change in the next interval is predictable (AUC 0.732); this matters for anomaly scoring even when MAE does not move.


## 10 · Saved report (never overwritten) and the SHARE BLOCK

In [ ]:
ALL = [(r[0], r[1], r[2], r[3]) for r in ER]; ok = all(r[2] for r in ALL)
rep = {'verdict_headroom': HEAD, 'message': MSG, 'change_occurrence': OCC, 'auc': AUC, 'tests': ALL, 'node_stats_by_rank': STAT_DF.to_dict('records'), 'binned': BIN, 'shared': {k: {kk: vv for kk, vv in v.items()} for k, v in SHR.items()}, 'shared_history': HIST, 'fit_windows': int(len(W_FIT)), 'edge': EDGE,
       'frozen': FROZEN, 'device': {'device': DEV, 'gpu': GPU}, 'test_days_opened': False, 'note': 'validation diagnostic; no frozen rule changed'}
pth = S.put_json(rep, OUT / 'd1b_report.json'); print('report:', pth.relative_to(BASE))
f = lambda v: f'{v:+.2f}%'
L = ['==== D1b SHARE BLOCK (copy everything between the two ==== lines) ====', f'verdict: {HEAD} | tests passed {sum(r[2] for r in ALL)}/{len(ALL)} | gpu {GPU} | test days opened: NO | lib {LIB_SHA[:12]} / {DLIB_SHA[:12]}', MSG,
     f'change occurrence: {OCC} (AUC {AUC:.3f}, base rate {SHR["pooled"]["base_rate_change"]:.3f})']
for nm in STAT_DF.index: r = STAT_DF.loc[nm]; L.append(f'  {r["ranks (bid and ask averaged)"]}: P(change) {r["P(change at t+1)"]:.3f}' + (f' | P(change|prev change) {r["P(change | changed at t)"]:.3f} | P(change|no prev) {r["P(change | no change at t)"]:.3f} | P(reverse|both) {r["P(reverses | both changed)"]:.3f}' if not np.isnan(r.get('P(change | changed at t)', np.nan)) else ''))
for k in VAL: L.append(f'  {k}: binned gain all {f(BIN[k]["rel_all_pct"])} changed {f(BIN[k]["rel_changed_pct"])} spread {f(BIN[k]["rel_spread_pct"])} | shared-net gain all {f(SHR[k]["rel_all_pct"])} changed {f(SHR[k]["rel_changed_pct"])} spread {f(SHR[k]["rel_spread_pct"])} | AUC {SHR[k]["auc_change"]:.3f}')
L.append('  by rank (shared net, pooled): ' + '; '.join(f'{k} {v["rel_pct"]:+.2f}%' for k, v in SHR['pooled']['by_rank'].items()))
L.append('failed tests: ' + (', '.join(r[0] for r in ALL if not r[2]) or 'none')); L.append('==== END ====')
print('\n'.join(L)); (OUT / 'd1b_share_block.txt').write_text('\n'.join(L)) if not (OUT / 'd1b_share_block.txt').exists() else None

report: TGNN_StageD_v1/d1b/d1b_report.json
==== D1b SHARE BLOCK (copy everything between the two ==== lines) ====
verdict: SMALL HEADROOM | tests passed 18/18 | gpu Tesla T4 | test days opened: NO | lib 85e7c1f444a4 / aade83abf736
The best simple model gains 0.84% (all targets) and 2.37% (changed targets). A 5% pass needs the TGNN to beat these simple models clearly.
change occurrence: predictable (AUC 0.732, base rate 0.219)
  rank 1-3: P(change) 0.180 | P(change|prev change) 0.415 | P(change|no prev) 0.128 | P(reverse|both) 0.595
  rank 4-10: P(change) 0.176 | P(change|prev change) 0.413 | P(change|no prev) 0.126 | P(reverse|both) 0.707
  rank 11-25: P(change) 0.251 | P(change|prev change) 0.446 | P(change|no prev) 0.186 | P(reverse|both) 0.729
  spread: P(change) 0.003
  pooled: binned gain all -0.02% changed +0.20% spread +16.63% | shared-net gain all +0.84% changed +2.37% spread +27.50% | AUC 0.732
  BTCUSDT_2024-07-01: binned gain all +0.28% changed +0.64% spread +18.89% | shared

1282